# Лабораторная 2 (GPU-вариант): декодерный трансформер на `Tiny Shakespeare`

Цель работы: выполнить расширенный перенос на реальный корпус в режиме `GPU-friendly`,
увеличить масштаб модели и подтвердить улучшение относительно `CPU`-варианта.

## Что нужно знать до старта
- Полностью завершён `CPU`-вариант `ЛР02` с зафиксированным `test_perplexity`.
- GPU-среда проходит `gpu_preflight()` без fallback и нестабильных kernels.
- Контракт gate-критериев остаётся тем же: сравнение с baseline + generation checks.

## Интуиция задачи без формул
GPU-вариант не меняет математику задачи, а масштабирует обучение в пределах зафиксированного бюджета.
Если `gpu_preflight`, causal-mask и leakage checks выполнены, улучшение качества объясняется обучением модели, а не сменой условий эксперимента.

## Контракт данных (быстрый ориентир)
- `token_windows` и `targets`: формы `(batch, context)` как в CPU-треке.
- `padding_mask`: исключает `PAD` из loss/metric.
- `causal_mask`: строго нижнетреугольная, без доступа к будущим позициям.
- `gpu_preflight`: обязательный preflight до длинного обучения, без скрытого CPU fallback.
- Итоговые проверки: baseline/perplexity/generation/leakage + индикатор `CPU_REFERENCE_PERPLEXITY`.



## Beginner-слой: как читать эту тетрадь с нуля

### Кому читать
- Если вы стартуете с нуля и хотите понимать, зачем каждый блок идёт именно в таком порядке.
- Если формулы пока тяжёлые, а опора нужна через интуицию и ручной мини-пример.

### Что изменилось после прошлого шага
- После 03-Transformer: понятны self-attention и роль positional/масок.

### Теоретический ориентир
- Теория этой темы: [../theory/theory.md](../theory/theory.md)
- Общий вход курса: [../../00-Foundations/README.md](../../00-Foundations/README.md)

Поток изучения: контракт -> теория -> ручной пример -> TODO -> проверки -> диагностика.


## Маршрут выполнения

Строгий порядок шагов:
1. Завершить `CPU`-вариант `ЛР02` и зафиксировать `test_perplexity`.
2. `TODO 1`: загрузка корпуса и построение словаря.
3. `TODO 2`: окна фиксированной длины и индексное разбиение.
4. `TODO 3`: декодерный блок с причинной маской.
5. `TODO 4`: обучение, метрики и сравнение с частотным ориентиром.
6. `TODO 5`: детерминированная генерация по фиксированным подсказкам.
7. `TODO 6`: диагностика внимания без доступа в будущее.


In [ ]:
import ctypes
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

RUNTIME_MODE = os.environ.get("COURSE_RUNTIME_MODE", "local-gpu")
COURSE_REPO_HTTPS_URL = os.environ.get(
    "COURSE_REPO_HTTPS_URL",
    "https://github.com/<org>/<repo>.git",
)
NOTEBOOK_REQUIREMENTS = "themes/04-Autoregression/lab/requirements.txt"


def _prepend_path_env(var_name: str, new_paths: list[Path]) -> None:
    """Добавляет пути в начало переменной окружения с путями.

    Аргументы:
      var_name: Имя переменной окружения (например, `LD_LIBRARY_PATH`).
      new_paths: Пути-кандидаты, которые нужно добавить в начало.

    Возвращает:
      `None`.

    Исключения:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    existing = os.environ.get(var_name, "")
    cleaned_new = []
    for path in new_paths:
        if path.is_dir():
            cleaned_new.append(str(path))

    if not cleaned_new:
        return

    existing_parts = [part for part in existing.split(":") if part]
    merged = []
    for part in cleaned_new + existing_parts:
        if part not in merged:
            merged.append(part)
    os.environ[var_name] = ":".join(merged)


def _detect_site_packages_dir() -> Path | None:
    """Находит каталог `site-packages` активной виртуальной среды.

    Аргументы:
      Нет.

    Возвращает:
      Путь к `site-packages` или `None`, если каталог не найден.

    Исключения:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    major, minor = sys.version_info[:2]
    candidate = Path(sys.prefix) / "lib" / f"python{major}.{minor}" / "site-packages"
    if candidate.is_dir():
        return candidate
    return None


def _preload_cuda_runtime_libraries(site_packages: Path) -> dict:
    """Предзагружает CUDA-библиотеки в текущий процесс до импорта TensorFlow.

    Аргументы:
      site_packages: Каталог `site-packages` активной виртуальной среды.

    Возвращает:
      Сводка со списками успешно загруженных, отсутствующих и проблемных библиотек.

    Исключения:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    nvidia_root = site_packages / "nvidia"
    library_specs = [
        ("cuda_runtime", "libcudart.so.12"),
        ("cublas", "libcublas.so.12"),
        ("cublas", "libcublasLt.so.12"),
        ("cudnn", "libcudnn.so.9"),
        ("cufft", "libcufft.so.11"),
        ("curand", "libcurand.so.10"),
        ("cusolver", "libcusolver.so.11"),
        ("cusparse", "libcusparse.so.12"),
        ("nccl", "libnccl.so.2"),
        ("nvjitlink", "libnvJitLink.so.12"),
    ]

    loaded = []
    missing = []
    failed = []

    for subdir, library_name in library_specs:
        library_path = nvidia_root / subdir / "lib" / library_name
        if not library_path.is_file():
            missing.append(str(library_path))
            continue
        try:
            ctypes.CDLL(str(library_path), mode=ctypes.RTLD_GLOBAL)
            loaded.append(str(library_path))
        except OSError as exc:
            failed.append(f"{library_path}: {exc}")

    return {
        "loaded": loaded,
        "missing": missing,
        "failed": failed,
    }


def _configure_local_gpu_runtime_env(runtime_mode: str) -> dict:
    """Готовит переменные окружения для локального запуска TensorFlow на GPU.

    Аргументы:
      runtime_mode: Запрошенный режим выполнения тетради.

    Возвращает:
      Словарь с краткой сводкой применённой настройки путей.

    Исключения:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    if runtime_mode != "local-gpu":
        return {
            "applied": False,
            "reason": "runtime_mode != local-gpu",
        }

    site_packages = _detect_site_packages_dir()
    if site_packages is None:
        return {
            "applied": False,
            "reason": "site-packages not found",
        }

    tensorflow_dir = site_packages / "tensorflow"
    nvidia_root = site_packages / "nvidia"
    nvidia_lib_dirs = sorted(path for path in nvidia_root.glob("*/lib") if path.is_dir())
    _prepend_path_env("LD_LIBRARY_PATH", [tensorflow_dir, *nvidia_lib_dirs])

    nvcc_bin = nvidia_root / "cuda_nvcc" / "bin"
    _prepend_path_env("PATH", [nvcc_bin])

    preload_report = _preload_cuda_runtime_libraries(site_packages)

    return {
        "applied": True,
        "tensorflow_dir": str(tensorflow_dir),
        "nvidia_lib_dirs": [str(path) for path in nvidia_lib_dirs],
        "nvcc_bin": str(nvcc_bin),
        "preload_report": preload_report,
    }


gpu_env_info = _configure_local_gpu_runtime_env(RUNTIME_MODE)
print("gpu_env_info:", gpu_env_info)


def _detect_notebook_platform():
    """Определяет тип среды выполнения текущей тетради.

    Аргументы:
      Нет.

    Возвращает:
      Строка из множества `{'local', 'colab', 'kaggle'}`.

    Исключения:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle").exists():
        return "kaggle"
    if os.environ.get("COLAB_RELEASE_TAG") or "google.colab" in sys.modules:
        return "colab"
    return "local"


def _looks_like_repo_root(path: Path) -> bool:
    """Проверяет, похож ли путь на корень учебного репозитория.

    Аргументы:
      path: Проверяемый путь.

    Возвращает:
      `True`, если обнаружены ключевые признаки корня репозитория.

    Исключения:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    return (
        path.is_dir()
        and (path / "themes").is_dir()
        and (path / "course_runtime.py").is_file()
    )


def _canonical_cloud_repo_root(platform: str) -> Path:
    """Возвращает стандартный путь клонирования для облачной платформы.

    Аргументы:
      platform: Имя платформы (`'colab'` или `'kaggle'`).

    Возвращает:
      Абсолютный путь каталога репозитория.

    Исключения:
      ValueError: Если передано неподдерживаемое имя платформы.
    """
    if platform == "colab":
        return Path("/content/students-AI_math_essentials")
    if platform == "kaggle":
        return Path("/kaggle/working/students-AI_math_essentials")
    raise ValueError(f"Unexpected cloud platform: {platform}")


def _is_placeholder_repo_url(repo_url: str) -> bool:
    """Проверяет, остался ли в настройке шаблонный URL репозитория.

    Аргументы:
      repo_url: Проверяемый URL репозитория.

    Возвращает:
      `True`, если URL имеет вид шаблона-заглушки.

    Исключения:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    return repo_url.strip() == "https://github.com/<org>/<repo>.git"


def _find_repo_root_from_cwd():
    """Ищет корень курса, поднимаясь от текущего каталога вверх.

    Аргументы:
      Нет.

    Возвращает:
      Объект `Path` корня репозитория или `None`, если путь не найден.

    Исключения:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if _looks_like_repo_root(candidate):
            return candidate
    return None


def _ensure_course_runtime_importable(runtime_mode: str, repo_url: str) -> None:
    """Обеспечивает доступность модуля `course_runtime` для текущей среды.

    Аргументы:
      runtime_mode: Режим запуска тетради.
      repo_url: URL репозитория курса для облачной автозагрузки.

    Возвращает:
      `None`.

    Исключения:
      ModuleNotFoundError: Если локальный запуск выполнен вне корректного корня репозитория.
      RuntimeError: Если в облаке отсутствует валидный URL репозитория или каталог повреждён.
      subprocess.CalledProcessError: Если команда `git clone` завершается с ошибкой.
    """
    if importlib.util.find_spec("course_runtime") is not None:
        return

    local_repo_root = _find_repo_root_from_cwd()
    if local_repo_root is not None:
        if str(local_repo_root) not in sys.path:
            sys.path.insert(0, str(local_repo_root))
        return

    platform = _detect_notebook_platform()
    if platform == "local":
        raise ModuleNotFoundError(
            "Не удалось импортировать course_runtime.py. Для локального запуска "
            "открывайте репозиторий через `.venv/bin/jupyter notebook` из корня проекта."
        )

    repo_root = _canonical_cloud_repo_root(platform)
    if not _looks_like_repo_root(repo_root):
        if _is_placeholder_repo_url(repo_url):
            raise RuntimeError(
                "Для cloud auto-bootstrap нужен публичный HTTPS URL курса. "
                "Замените COURSE_REPO_HTTPS_URL на реальный адрес репозитория."
            )
        repo_root.parent.mkdir(parents=True, exist_ok=True)
        if repo_root.exists() and any(repo_root.iterdir()):
            raise RuntimeError(
                f"Каталог {repo_root} уже существует, но не выглядит как корень курса. "
                "Очистите runtime или используйте новый notebook session."
            )
        print(f"Bootstrapping course repository into {repo_root} ...")
        subprocess.run(["git", "clone", repo_url, str(repo_root)], check=True)

    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))


_ensure_course_runtime_importable(RUNTIME_MODE, COURSE_REPO_HTTPS_URL)

from course_runtime import setup_notebook_runtime

runtime_info = setup_notebook_runtime(
    runtime_mode=RUNTIME_MODE,
    course_repo_https_url=COURSE_REPO_HTTPS_URL,
    notebook_requirements=NOTEBOOK_REQUIREMENTS,
)
runtime_info.as_dict()


## Константы GPU-варианта

Этот трек выполняется только при наличии графического процессора.
Если `GPU` недоступен, работу по этой тетради нужно перенести в среду с корректно установленными CUDA-драйверами.

Перед запуском установите `COURSE_RUNTIME_MODE=local-gpu`.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from pathlib import Path

SEED = 29
PAD_ID = 0
CHECK_GEN_STEPS = 16
PROMPT_COUNT = 20
CPU_REFERENCE_PERPLEXITY = 7.64

GPU_60M_PROFILE = {
    'chars': 900_000,
    'context': 128,
    'stride': 1,
    'batch_size': 128,
    'embed_dim': 192,
    'num_heads': 6,
    'ff_dim': 384,
    'eval_every_steps': 1200,
    'validation_steps': 64,
    'max_eval_rounds': 180,
    'early_stopping_patience': 16,
    'early_stopping_min_delta': 1e-4,
    'early_stopping_probe_floor': 18,
    'min_minutes_before_early_stop': 45,
    'max_training_minutes': 60,
    'min_minutes_before_generation_stop': 20,
    'generation_stop_patience': 2,
    'generation_probe_every': 4,
    'learning_rate': 1e-3,
    'gen_match_ratio': 0.18,
    'gen_threshold': 19,
    'gen_mean_threshold': 0.25,
}

GPU_60M_PROFILE_BOOST = {
    **GPU_60M_PROFILE,
    'eval_every_steps': 1500,
    'validation_steps': 64,
    'generation_probe_every': 3,
    'early_stopping_patience': 20,
    'learning_rate': 8e-4,
}

GPU_PROFILE_MAP = {
    'gpu_60m': GPU_60M_PROFILE,
    'gpu_60m_boost': GPU_60M_PROFILE_BOOST,
}


def gpu_preflight(runtime_info):
    """Проверяет готовность локального GPU-контура перед длинным обучением.

    Аргументы:
      runtime_info: Результат `setup_notebook_runtime(...)` с полем `effective_mode`.

    Возвращает:
      Словарь с краткой сводкой по обнаруженным GPU и версии TensorFlow.

    Исключения:
      RuntimeError: Если режим запуска не `local-gpu`, GPU не виден,
        либо GPU-ядра нестабильны на реальной короткой нагрузке.
    """
    effective_mode = getattr(runtime_info, 'effective_mode', '')
    print('effective_mode:', effective_mode)
    if effective_mode != 'local-gpu':
        raise RuntimeError(
            'GPU-вариант ЛР02 должен запускаться только в режиме local-gpu. '
            'Установите COURSE_RUNTIME_MODE=local-gpu и перезапустите ядро.'
        )

    try:
        nvidia_report = subprocess.run(
            ['nvidia-smi', '-L'],
            check=True,
            capture_output=True,
            text=True,
        )
        lines = [line for line in nvidia_report.stdout.strip().splitlines() if line.strip()]
        print('nvidia-smi -L (первые строки):')
        for line in lines[:3]:
            print('  ', line)
    except Exception as exc:
        raise RuntimeError(
            'GPU не виден (setup): команда nvidia-smi недоступна или вернула ошибку. '
            'Используйте уже подготовленную среду из '
            'themes/00-Foundations/guides/05_local_tensorflow_gpu_notebooks.md '
            'и не переустанавливайте ОС внутри ЛР.'
        ) from exc

    print('TensorFlow version:', tf.__version__)
    print('TensorFlow built with CUDA:', tf.test.is_built_with_cuda())
    build_info = tf.sysconfig.get_build_info()
    print('TensorFlow build cuda_version:', build_info.get('cuda_version', 'unknown'))
    print('TensorFlow build cudnn_version:', build_info.get('cudnn_version', 'unknown'))

    physical_gpus = tf.config.list_physical_devices('GPU')
    logical_gpus = tf.config.list_logical_devices('GPU')
    print('Physical GPUs:', [device.name for device in physical_gpus])
    print('Logical GPUs :', [device.name for device in logical_gpus])

    if len(physical_gpus) == 0 or len(logical_gpus) == 0:
        raise RuntimeError(
            'GPU не виден (setup): TensorFlow не зарегистрировал GPU-устройства. '
            'Проверьте окружение по guide 05/06 в themes/00-Foundations.'
        )

    try:
        # Проверка 1: базовая матричная операция на /GPU:0.
        with tf.device('/GPU:0'):
            lhs = tf.random.normal((128, 128), dtype=tf.float32)
            rhs = tf.random.normal((128, 128), dtype=tf.float32)
            product = tf.matmul(lhs, rhs)
            _ = float(tf.reduce_mean(product).numpy())

        # Проверка 2: короткий notebook-like шаг обучения Keras на /GPU:0.
        features = np.random.normal(size=(32, 8)).astype('float32')
        targets = np.random.normal(size=(32, 1)).astype('float32')
        with tf.device('/GPU:0'):
            smoke_model = keras.Sequential(
                [
                    layers.Input(shape=(8,)),
                    layers.Dense(16, activation='relu'),
                    layers.Dense(1),
                ],
                name='gpu_smoke_model',
            )
            smoke_model.compile(
                optimizer=keras.optimizers.Adam(1e-3),
                loss='mse',
            )
            smoke_model.train_on_batch(features, targets)
    except Exception as exc:
        raise RuntimeError(
            'GPU виден, но kernels падают (compatibility). Это не ошибка кода ЛР. '
            'См. themes/00-Foundations/guides/07_tensorflow_blackwell_local_gpu_case_study.md. '
            f'Исходная ошибка: {type(exc).__name__}: {exc}'
        ) from exc

    print('gpu_preflight(): PASSED')
    return {
        'tensorflow_version': tf.__version__,
        'cuda_version': build_info.get('cuda_version', 'unknown'),
        'cudnn_version': build_info.get('cudnn_version', 'unknown'),
        'physical_gpus': [device.name for device in physical_gpus],
        'logical_gpus': [device.name for device in logical_gpus],
    }


RUN_MODE = 'gpu'
selected_profile_name = os.environ.get('GPU_PROFILE_NAME', 'gpu_60m').strip().lower()
if selected_profile_name not in GPU_PROFILE_MAP:
    raise ValueError(
        f"Неизвестный профиль GPU: {selected_profile_name}. "
        f"Допустимые значения: {sorted(GPU_PROFILE_MAP)}"
    )

cfg = dict(GPU_PROFILE_MAP[selected_profile_name])
cfg['max_training_minutes'] = float(
    os.environ.get('GPU_TRAINING_BUDGET_MINUTES', cfg['max_training_minutes'])
)
cfg['warmup_steps'] = int(os.environ.get('GPU_WARMUP_STEPS', '2'))
cfg['warmup_probe_steps'] = int(os.environ.get('GPU_WARMUP_PROBE_STEPS', '2'))

if cfg['max_training_minutes'] <= 0:
    raise ValueError('GPU_TRAINING_BUDGET_MINUTES должен быть положительным числом.')
if cfg['warmup_steps'] < 1:
    raise ValueError('GPU_WARMUP_STEPS должен быть целым числом >= 1.')
if cfg['warmup_probe_steps'] < 1:
    raise ValueError('GPU_WARMUP_PROBE_STEPS должен быть целым числом >= 1.')

plt.style.use('default')
keras.utils.set_random_seed(SEED)

preflight_info = gpu_preflight(runtime_info)
print('Режим выполнения:', RUN_MODE)
print('Профиль GPU:', selected_profile_name)
print('Измеряемый бюджет обучения (мин):', cfg['max_training_minutes'])


## Математический ориентир

Мы оптимизируем токенную перекрёстную энтропию (cross-entropy), которая эквивалентна
среднему отрицательному лог-правдоподобию для next-token задачи.

Перплексия (perplexity) вычисляется как:

$$
\mathrm{PPL} = e^{\mathrm{loss}}
$$

Для режима `gpu` дополнительно проверяется улучшение относительно
ориентира `CPU_REFERENCE_PERPLEXITY`.


## TODO 1: загрузка корпуса и построение словаря


In [ ]:
def load_tiny_shakespeare(profile_cfg):
    """Загружает корпус и строит детерминированное символьное кодирование.

    Аргументы:
      profile_cfg: Словарь параметров выбранного профиля.

    Возвращает:
      Кортеж `(text, encoded_ids, vocab, char_to_id, id_to_char)`.

    Исключения:
      ValueError: Если выбранный профиль задаёт слишком короткий фрагмент текста.
    """
    # TODO 1.1: Загрузите корпус Tiny Shakespeare через tf.keras.utils.get_file.
    # TODO 1.2: Возьмите детерминированный срез длины profile_cfg['chars'].
    # TODO 1.3: Постройте словарь ['<PAD>', ...] и кодирование в int32.
    raise NotImplementedError('TODO 1: реализуйте load_tiny_shakespeare')


# TODO 1.4: Вызовите load_tiny_shakespeare(cfg) и сохраните результаты в
# text, encoded_ids, vocab, char_to_id, id_to_char.
raise NotImplementedError('TODO 1: выполните загрузку корпуса')


In [ ]:
# Мини-проверка после TODO 1
assert len(text) == cfg['chars'], 'Длина среза текста не совпадает с профилем.'
assert encoded_ids.dtype == np.int32, 'Кодирование должно быть int32.'
assert vocab[PAD_ID] == '<PAD>', 'Нулевой идентификатор должен быть зарезервирован под PAD.'

text_b, ids_b, _, _, _ = load_tiny_shakespeare(cfg)
assert np.array_equal(encoded_ids[:1000], ids_b[:1000]), 'Кодирование должно быть воспроизводимым.'

print('Режим выполнения:', RUN_MODE)
print('Длина текста:', len(text))
print('Размер словаря:', len(vocab))


## TODO 2: окна фиксированной длины и разбиение


In [ ]:
def build_windows(encoded_stream, context_len, stride):
    """Строит обучающие окна фиксированной длины.

    Аргументы:
      encoded_stream: Одномерный массив кодов символов.
      context_len: Длина входного контекста.
      stride: Шаг между соседними окнами.

    Возвращает:
      Кортеж `(X, Y, starts)`.

    Исключения:
      ValueError: Если поток слишком короткий для построения хотя бы одного окна.
    """
    # TODO 2.1: Постройте X и Y со сдвигом на один шаг.
    # TODO 2.2: Верните также массив стартовых индексов starts.
    raise NotImplementedError('TODO 2: реализуйте build_windows')


# TODO 2.3: Выполните индексное разбиение X_all, y_all, starts_all на
# train/val/test без случайного перемешивания.
raise NotImplementedError('TODO 2: реализуйте детерминированное разбиение')


In [ ]:
# Мини-проверка после TODO 2
assert X_train.shape[1] == cfg['context']
assert y_train.shape == X_train.shape
assert starts_train.ndim == 1

# Проверка сдвига внутри окон.
assert np.array_equal(X_train[0, 1:], y_train[0, :-1]), 'Сдвиг X/Y нарушен.'

print('Окон train/val/test:', len(X_train), len(X_val), len(X_test))


## TODO 3: декодерный блок с причинной маской


In [ ]:
def build_causal_mask(seq_len):
    """Строит нижнетреугольную причинную маску.

    Аргументы:
      seq_len: Длина последовательности.

    Возвращает:
      Булев тензор формы `(seq_len, seq_len)`.

    Исключения:
      tf.errors.InvalidArgumentError: Если `seq_len` не является положительным.
    """
    # TODO 3.1: Реализуйте построение нижнетреугольной маски. Используйте тензорную
    # проверку `tf.debugging.assert_positive`, чтобы код работал и в графовом режиме.
    raise NotImplementedError('TODO 3: реализуйте build_causal_mask')


class TokenAndPositionEmbedding(layers.Layer):
    """Складывает токенные и позиционные векторы.

    Аргументы:
      maxlen: Максимальная длина контекста.
      vocab_size: Размер словаря.
      embed_dim: Размер скрытого представления.
      **kwargs: Дополнительные аргументы базового слоя.

    Возвращает:
      Экземпляр слоя встраивания.

    Исключения:
      ValueError: Если `embed_dim` меньше 1.
    """

    def __init__(self, maxlen, vocab_size, embed_dim, **kwargs):
        """Инициализирует слой токенного и позиционного встраивания.

        Аргументы:
          maxlen: Максимальная длина контекста.
          vocab_size: Размер словаря токенов.
          embed_dim: Размерность векторного представления.
          **kwargs: Дополнительные аргументы базового слоя Keras.

        Возвращает:
          None.

        Исключения:
          ValueError: Если `embed_dim` меньше 1.
        """
        super().__init__(**kwargs)
        if embed_dim < 1:
            raise ValueError('embed_dim должен быть положительным.')
        self.token_embedding = layers.Embedding(vocab_size, embed_dim, mask_zero=True)
        self.position_embedding = layers.Embedding(maxlen, embed_dim)

    def call(self, inputs):
        """Суммирует токенные и позиционные векторы.

        Аргументы:
          inputs: Матрица идентификаторов формы `(batch, time)`.

        Возвращает:
          Тензор формы `(batch, time, embed_dim)`.

        Исключения:
          NotImplementedError: Пока шаг `TODO 3` не реализован.
        """
        # TODO 3.2: Реализуйте сложение token embedding и position embedding.
        raise NotImplementedError('TODO 3: реализуйте TokenAndPositionEmbedding.call')

    def compute_mask(self, inputs, mask=None):
        """Пробрасывает маску непустых токенов.

        Аргументы:
          inputs: Матрица токенов.
          mask: Входная маска.

        Возвращает:
          Булева маска формы `(batch, time)`.

        Исключения:
          RuntimeError: Не выбрасывается в штатном режиме.
        """
        return self.token_embedding.compute_mask(inputs)


class CausalDecoderBlock(layers.Layer):
    """Минимальный декодерный блок с причинной маской.

    Аргументы:
      embed_dim: Размер скрытого представления.
      num_heads: Число голов внимания.
      ff_dim: Размер скрытого слоя позиционно-независимой сети.
      rate: Доля прореживания.
      **kwargs: Дополнительные аргументы базового слоя.

    Возвращает:
      Экземпляр декодерного блока.

    Исключения:
      ValueError: Если `embed_dim` не делится на `num_heads`.
    """

    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kwargs):
        """Создаёт внутренние слои декодерного блока.

        Аргументы:
          embed_dim: Размерность входных признаков.
          num_heads: Число голов внимания.
          ff_dim: Размер скрытого слоя позициионно-независимой сети.
          rate: Доля выключаемых нейронов в прореживании.
          **kwargs: Дополнительные аргументы базового слоя Keras.

        Возвращает:
          None.

        Исключения:
          ValueError: Если `embed_dim` не делится на `num_heads`.
        """
        super().__init__(**kwargs)
        if embed_dim % num_heads != 0:
            raise ValueError('embed_dim должен делиться на num_heads без остатка.')
        self.self_attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=rate,
        )
        self.ffn = keras.Sequential(
            [
                layers.Dense(ff_dim, activation='relu'),
                layers.Dense(embed_dim),
            ]
        )
        self.norm_1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm_2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout_1 = layers.Dropout(rate)
        self.dropout_2 = layers.Dropout(rate)

    def call(self, inputs, padding_mask=None, training=None, return_attention_scores=False):
        """Прогоняет вход через маскированное самовнимание и FFN.

        Аргументы:
          inputs: Тензор формы `(batch, time, embed_dim)`.
          padding_mask: Булева маска формы `(batch, time)`.
          training: Признак режима обучения.
          return_attention_scores: Нужно ли вернуть веса внимания.

        Возвращает:
          Либо выходной тензор, либо кортеж `(выход, attention_scores)`.

        Исключения:
          NotImplementedError: Пока шаг `TODO 3` не реализован.
        """
        # TODO 3.3: Реализуйте causal mask + padding mask и прямой проход блока.
        raise NotImplementedError('TODO 3: реализуйте CausalDecoderBlock.call')

    def compute_mask(self, inputs, mask=None):
        """Отключает автоматическое пробрасывание маски в выход блока.

        Аргументы:
          inputs: Входной тензор признаков.
          mask: Маска, пришедшая от предыдущего слоя.

        Возвращает:
          `None`, чтобы маскирование контролировалось явно в функции потерь.

        Исключения:
          RuntimeError: Не выбрасывается в штатном режиме.
        """
        return None


def masked_sparse_crossentropy(y_true, y_pred):
    """Считает перекрёстную энтропию по непустым позициям.

    Аргументы:
      y_true: Истинные токены.
      y_pred: Предсказанные распределения.

    Возвращает:
      Среднее значение функции потерь.

    Исключения:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    per_token = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
    mask = tf.cast(tf.not_equal(y_true, PAD_ID), tf.float32)
    return tf.reduce_sum(per_token * mask) / tf.reduce_sum(mask)


def masked_token_accuracy(y_true, y_pred):
    """Считает токенную точность по непустым позициям.

    Аргументы:
      y_true: Истинные токены.
      y_pred: Предсказанные распределения.

    Возвращает:
      Доля верных токенов.

    Исключения:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    pred = tf.argmax(y_pred, axis=-1, output_type=y_true.dtype)
    correct = tf.cast(tf.equal(y_true, pred), tf.float32)
    mask = tf.cast(tf.not_equal(y_true, PAD_ID), tf.float32)
    return tf.reduce_sum(correct * mask) / tf.reduce_sum(mask)


def perplexity_from_loss(loss_value):
    """Преобразует значение функции потерь в перплексию.

    Аргументы:
      loss_value: Средняя перекрёстная энтропия.

    Возвращает:
      Значение перплексии.

    Исключения:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    return float(np.exp(loss_value))


def frequency_baseline_perplexity(y_train_data, y_test_data, vocab_size, pad_id=PAD_ID):
    """Считает частотный базовый ориентир перплексии.

    Аргументы:
      y_train_data: Целевые токены обучающей выборки.
      y_test_data: Целевые токены тестовой выборки.
      vocab_size: Размер словаря.
      pad_id: Идентификатор дополнения.

    Возвращает:
      Значение базовой перплексии.

    Исключения:
      ValueError: Если в данных нет полезных токенов.
    """
    train_tokens = y_train_data[y_train_data != pad_id].reshape(-1)
    test_tokens = y_test_data[y_test_data != pad_id].reshape(-1)
    if len(train_tokens) == 0 or len(test_tokens) == 0:
        raise ValueError('Для базового ориентира нужны непустые токены.')

    counts = np.bincount(train_tokens, minlength=vocab_size).astype(np.float64)
    probs = counts / counts.sum()
    probs = np.maximum(probs, 1e-12)

    nll = -np.mean(np.log(probs[test_tokens]))
    return float(np.exp(nll))


## TODO 4: сборка и обучение модели

Обязательный контракт GPU-варианта:
- лимит времени `cfg['max_training_minutes']` минут (по умолчанию `90`);
- валидация каждые `cfg['eval_every_steps']` шагов;
- ранняя остановка по `val_loss`;
- целевая генерация: не ниже `19/20` фиксированных подсказок.


In [ ]:
# TODO 4.1: Соберите модель decoder-only с TokenAndPositionEmbedding и CausalDecoderBlock.
# TODO 4.2: Скомпилируйте модель с masked_sparse_crossentropy и masked_token_accuracy.
# TODO 4.3: Реализуйте отдельный warm-up этап (JIT/компиляция), который НЕ входит
#           в измеряемый бюджет обучения.
# TODO 4.4: После warm-up запустите измеряемый цикл обучения с лимитом
#           cfg['max_training_minutes'] минут (по умолчанию 60).
# TODO 4.5: Делайте проверку на валидации каждые cfg['eval_every_steps'] шагов.
# TODO 4.6: Generation-probe стройте на starts_val (валидация), а test оставьте
#           только для финальной оценки.
# TODO 4.7: Реализуйте контролируемую оценку continuation (teacher forcing):
#           сравнивайте предсказание следующего токена при эталонной истории,
#           чтобы избежать каскадного накопления ошибок в gate-критерии.
# TODO 4.8: Добавьте вспомогательную раннюю остановку по val_loss так, чтобы она не
#           срывала достижение жёсткой цели 19/20 преждевременно.
# TODO 4.9: Для остановки по generation-probe требуйте стабильное достижение цели:
#           минимум N подряд проверок и минимальное время после старта timed-run.
# TODO 4.10: Сформируйте training_trace со следующими ключами:
#           'step', 'train_loss', 'val_loss', 'val_token_accuracy',
#           'probe_success_count', 'probe_mean_match_ratio',
#           'round_train_seconds', 'round_eval_seconds', 'round_probe_seconds',
#           'round_total_seconds', 'stop_reason', 'elapsed_minutes',
#           'completed_steps', 'warmup_minutes', 'budget_minutes'.
# TODO 4.11: Посчитайте test_loss, test_accuracy, test_perplexity и baseline_perplexity.
#            Сравнение с CPU_REFERENCE_PERPLEXITY оформите как индикатор (без assert).
raise NotImplementedError('TODO 4: обучение до 19/20 в режиме warm-up + timed-run')


In [ ]:
# Графики и контроль после TODO 4
steps = training_trace['step']
train_losses = training_trace['train_loss']
val_losses = training_trace['val_loss']

plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
plt.plot(steps, train_losses, label='train_loss')
plt.plot(steps, val_losses, label='val_loss')
plt.xlabel('шаг обучения')
plt.ylabel('loss')
plt.legend()

plt.subplot(1, 2, 2)
train_ppl = [perplexity_from_loss(v) for v in train_losses]
val_ppl = [perplexity_from_loss(v) for v in val_losses]
plt.plot(steps, train_ppl, label='train_ppl')
plt.plot(steps, val_ppl, label='val_ppl')
plt.xlabel('шаг обучения')
plt.ylabel('perplexity')
plt.legend()
plt.tight_layout()

print(f"Причина остановки    : {training_trace['stop_reason']}")
print(f"Warm-up (мин)        : {training_trace['warmup_minutes']:.2f}")
print(f"Бюджет после warm-up : {training_trace['budget_minutes']:.2f}")
print(f"Измеряемое время (мин): {training_trace['elapsed_minutes']:.2f}")
print(f"Выполнено шагов      : {training_trace['completed_steps']}")
print(f"test_loss            : {test_loss:.4f}")
print(f"test_token_accuracy  : {test_token_accuracy:.4f}")
print(f"test_perplexity      : {test_perplexity:.4f}")
print(f"baseline_perplexity  : {baseline_perplexity:.4f}")
print(f"baseline_pass        : {baseline_pass}")
print(f"cpu_reference_pass   : {cpu_reference_pass}")


## TODO 5: детерминированная генерация по фиксированным подсказкам


In [ ]:
def ids_to_text(token_ids, id_to_char_map):
    """Преобразует идентификаторы в строку.

    Аргументы:
      token_ids: Последовательность идентификаторов.
      id_to_char_map: Отображение id -> символ.

    Возвращает:
      Строка символов.

    Исключения:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    return ''.join(id_to_char_map.get(int(token), '') for token in token_ids if int(token) != PAD_ID)


def generate_greedy(model, prompt_ids, steps, context_len):
    """Генерирует продолжение в режиме argmax через свободную автогенерацию.

    Аргументы:
      model: Обученная модель.
      prompt_ids: Начальная последовательность идентификаторов.
      steps: Число новых токенов.
      context_len: Длина модельного контекста.

    Возвращает:
      Список токенов продолжения длины не больше `steps`.

    Исключения:
      ValueError: Если подсказка пуста.
    """
    # TODO 5.1: Реализуйте генерацию argmax через прямой вызов
    # model(model_input, training=False), без model.predict(...).
    raise NotImplementedError('TODO 5: реализуйте generate_greedy')


def build_prompt_targets(encoded_stream, start_indices, context_len, continuation_len, n_prompts):
    """Готовит фиксированные подсказки и эталонные продолжения из тестового фрагмента.

    Аргументы:
      encoded_stream: Полный поток кодов корпуса.
      start_indices: Стартовые индексы окон тестовой части.
      context_len: Длина контекста модели.
      continuation_len: Длина целевого продолжения.
      n_prompts: Количество подсказок.

    Возвращает:
      Список пар `(prompt_ids, target_ids)`.

    Исключения:
      ValueError: Если тестовых окон недостаточно.
    """
    valid_starts = [
        int(start) for start in start_indices
        if int(start) + context_len + continuation_len <= len(encoded_stream)
    ]
    if len(valid_starts) < n_prompts:
        raise ValueError('Недостаточно тестовых окон для фиксированного набора подсказок.')

    selected = np.linspace(0, len(valid_starts) - 1, n_prompts, dtype=int)
    pairs = []
    for idx in selected:
        start = valid_starts[int(idx)]
        prompt = encoded_stream[start:start + context_len]
        target = encoded_stream[start + context_len:start + context_len + continuation_len]
        pairs.append((prompt.tolist(), target.tolist()))
    return pairs


# TODO 5.2: Посчитайте success_count и mean_match_ratio по контролируемому
# продолжению (teacher forcing) на фиксированных test-подсказках.
# Подсказка: на каждом шаге предсказывайте следующий токен при эталонной истории,
# а не при истории собственных ошибок модели.
# TODO 5.3: Свободную автогенерацию оставьте как демонстрационный блок примеров.
# TODO 5.4: Соберите run_summary и выведите строку
# `GPU_RUN_SUMMARY_JSON=<json>`, чтобы скрипт мог разобрать результат.
# TODO 5.5: Сравнение с CPU_REFERENCE_PERPLEXITY оставьте индикатором (без assert).
raise NotImplementedError('TODO 5: выполните проверку контролируемого продолжения и сформируйте run_summary')


## TODO 6: диагностика внимания


In [ ]:
# TODO 6.1: Постройте trace-модель, которая возвращает attention_scores из decoder_layer.
# TODO 6.2: Усредните веса внимания по головам и вычислите отношение массы в будущем.
# TODO 6.3: Проверьте, что отношение массы в будущем меньше 1e-4.
raise NotImplementedError('TODO 6: выполните диагностику внимания')


## Чек-лист перед завершением `ЛР02` (GPU-вариант)

1. Все `TODO` закрыты.
2. `gpu_preflight()` пройден полностью.
3. Выполнен отдельный warm-up; его время не входит в измеряемый бюджет.
4. Измеряемый бюджет после warm-up: `60` минут (`cfg['max_training_minutes']`).
5. Жёсткий критерий `19/20` считается по контролируемому продолжению (teacher forcing).
6. Дополнительно достигнут средний порог `mean_match_ratio` для контролируемого продолжения.
7. `test_perplexity < baseline_perplexity`.
8. `test_perplexity < CPU_REFERENCE_PERPLEXITY` — индикаторный контроль (без аварийного assert).
9. Свободная автогенерация остаётся демонстрационным блоком.
10. Диагностика внимания подтверждает отсутствие доступа в будущее.
